In [ ]:
"""
This block compares the former MIROC PD dataset (AmonZ, v2) and the extended MIROC PD dataset (monZ, v1) 
for equatorial zonal wind (ua).

Processing steps:
1. Load former (v2) and extended (v1) datasets separately.
2. Restrict the extended dataset to the same time range as the former dataset.
3. Interpolate the extended dataset onto the former dataset’s pressure grid.
4. Compute the difference (extended − former).
5. Produce four panels:
   - Former dataset (time–height)
   - Extended dataset (time–height)
   - Difference (extended − former)
   - Former as background with extended zero-wind contour overlay

Purpose:
To verify structural consistency between the original MIROC PD output and the newly extended simulation.
"""

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os

# 输出目录
out_dir = "/home/weiji/restart_exam/20250429/MIROC/attempt"
os.makedirs(out_dir, exist_ok=True)

# —— 1. 路径函数 —— #

def get_paths_v1(data_type, var_name):
    base = f"/mnt/backup_ETH/MIROC-ES2H/PD-{data_type}/r1i1p1f1/monZ/{var_name}/gn/v20241223/"
    if data_type == 'NINT':
        segments = [
            ('r1i1p1f1', '200001', '202912'),
            ('r2i1p1f1', '203001', '205912'),
            ('r3i1p1f1', '206001', '208912'),
        ]
        return [
            f"{base}{var_name}_monZ_MIROC-ES2H_PDNINT_{m}_gn_{s}-{e}.nc"
            for m, s, e in segments
        ]
    else:
        return [f"{base}{var_name}_monZ_MIROC-ES2H_PDINT_r1i1p1f1_gn_200001-208912.nc"]

def get_paths_v2(data_type, var_name):
    base1 = f"/mnt/backup_ETH/MIROC-ES2H/PD-{data_type}/r1i1p1f1/AmonZ/{var_name}/gn/v20240220/"
    base2 = f"/mnt/backup_ETH/MIROC-ES2H/PD-{data_type}/r1i1p1f1/AmonZ/{var_name}/gn/v20240517/"
    return [
        f"{base1}{var_name}_AmonZ_MIROC-ES2H_PD-{data_type}_r1i1p1f1_gn_202001-206912.nc",
        f"{base2}{var_name}_AmonZ_MIROC-ES2H_PD-{data_type}_r1i1p1f1_gn_207001-207412.nc"
    ]

# —— 2. 数据处理函数 —— #

def process_data(var_name, paths, lat_range=(-5,5), deseasoned=False):
    ds_list = [xr.open_dataset(p)[var_name] for p in paths]
    data = xr.concat(ds_list, dim='time')
    data_eq = data.sel(lat=slice(*lat_range)).mean(dim='lat')
    if deseasoned:
        clim = data_eq.groupby('time.month').mean('time')
        data_eq = data_eq.groupby('time.month') - clim
    tmin = data_eq.time.min().values
    tmax = data_eq.time.max().values
    print(f"  → [{var_name}] time range: {np.datetime_as_string(tmin, unit='D')} to {np.datetime_as_string(tmax, unit='D')}")
    years = data_eq.time.dt.year + data_eq.time.dt.month/12
    plev_hPa = data_eq.plev / 100.0
    for ds in ds_list:
        ds.close()
    return data_eq, years, plev_hPa

# —— 3. 主循环：INT & NINT —— #

for data_type in ['INT', 'NINT']:
    print(f"\n** Processing {data_type} **")
    var = 'ua'

    # former MIROC (v2)
    print(" Loading former MIROC (v2):")
    data_v2, years_v2, plev_hPa = process_data(var, get_paths_v2(data_type, var))

    # extended MIROC (v1)
    print(" Loading extended MIROC (v1):")
    data_v1_full, _, _ = process_data(var, get_paths_v1(data_type, var))

    print("  v2 dims, shape:", data_v2.dims, data_v2.shape)
    print("  v1_full dims, shape:", data_v1_full.dims, data_v1_full.shape)

    # 对齐时间
    t0 = data_v2.time.min().values
    t1 = data_v2.time.max().values
    print(f" Slicing v1 to v2 time range: {np.datetime_as_string(t0, unit='D')} to {np.datetime_as_string(t1, unit='D')}")
    data_v1_slice = data_v1_full.sel(time=slice(t0, t1))
    print("  v1_slice dims, shape:", data_v1_slice.dims, data_v1_slice.shape)

    # 插值到相同 plev 网格
    print(" Interpolating v1_slice to v2 plev grid:")
    data_v1 = data_v1_slice.interp(plev=data_v2.plev)
    years_v1 = data_v1.time.dt.year + data_v1.time.dt.month/12
    print("  v1_interp dims, shape:", data_v1.dims, data_v1.shape)

    # 差值
    data_diff = data_v1 - data_v2
    print("  diff dims, shape:", data_diff.dims, data_diff.shape)

    # 等高线参数
    abs_levels = np.linspace(-40, 40, 21)

    # —— 4. 绘图 —— #
    fig = plt.figure(figsize=(24,16))
    gs = gridspec.GridSpec(4, 5,
                           width_ratios=[1,1,1,1,0.05],
                           height_ratios=[1,1,1,1],
                           wspace=0.3, hspace=0.4)

    ax0 = fig.add_subplot(gs[0, :4])
    ax1 = fig.add_subplot(gs[1, :4], sharex=ax0, sharey=ax0)
    ax2 = fig.add_subplot(gs[2, :4], sharex=ax0, sharey=ax0)
    ax3 = fig.add_subplot(gs[3, :4], sharex=ax0, sharey=ax0)

    # 子图1：former MIROC
    cf0 = ax0.contourf(years_v2, plev_hPa, data_v2.T,
                       levels=abs_levels, cmap='RdBu_r', extend='both')
    ax0.contour(years_v2, plev_hPa, data_v2.T, levels=[0], colors='k')
    ax0.set_title(f'{data_type}: former MIROC data')
    ax0.set_yscale('log'); ax0.invert_yaxis()
    ax0.set_yticks([1,5,10,20,50,100]); ax0.set_ylim(100,1)
    ax0.grid(True, alpha=0.3)

    # 子图2：extended MIROC
    cf1 = ax1.contourf(years_v1, plev_hPa, data_v1.T,
                       levels=abs_levels, cmap='RdBu_r', extend='both')
    ax1.contour(years_v1, plev_hPa, data_v1.T, levels=[0], colors='k')
    ax1.set_title(f'{data_type}: extended MIROC data')
    ax1.grid(True, alpha=0.3)

    # 子图3：difference (extended – former)
    cf2 = ax2.contourf(years_v2, plev_hPa, data_diff.T,
                       levels=abs_levels, cmap='RdBu_r', extend='both')
    ax2.contour(years_v2, plev_hPa, data_diff.T, levels=[0], colors='k')
    ax2.set_title(f'{data_type}: difference (extended – former)')
    ax2.set_xlabel('Year'); ax2.set_ylabel('Pressure (hPa)')
    ax2.grid(True, alpha=0.3)

    # 子图4：background + extended overlay with hatches based on extended data sign
    cf3 = ax3.contourf(years_v2, plev_hPa, data_v2.T,
                       levels=abs_levels, cmap='RdBu_r', extend='both')
    # overlay extended zero contour
    ax3.contour(years_v1, plev_hPa, data_v1.T,
                levels=[0], colors='green', linestyles='--', linewidths=3)
    # hatch where extended > 0 (left slash)
#    ext_max = np.nanmax(np.abs(data_v1))
#    ax3.contourf(years_v1, plev_hPa, data_v1.T,
#                 levels=[0, ext_max],
#                 colors='none', hatches=['/'], linewidths=0)
    # hatch where extended < 0 (right slash)
#    ax3.contourf(years_v1, plev_hPa, data_v1.T,
#                 levels=[-ext_max, 0],
#                 colors='none', hatches=['\\\\'], linewidths=0)
    ax3.set_title(f'{data_type}: former base + extended overlay\n(green-dashed=extended positive westward wind)')
    ax3.set_xlabel('Year'); ax3.set_ylabel('Pressure (hPa)')
    ax3.grid(True, alpha=0.3)

    # —— 5. 两个独立 colorbar —— #
    cax_abs = fig.add_subplot(gs[0:2, 4])
    fig.colorbar(cf1, cax=cax_abs, label='Eastward Wind (m/s)')
    cax_diff = fig.add_subplot(gs[2, 4])
    fig.colorbar(cf2, cax=cax_diff, label='Difference (m/s)')

    # 保存
    fname = os.path.join(out_dir, f'comparison_eastward_wind_{data_type}.png')
    plt.savefig(fname, dpi=300, bbox_inches='tight')
    plt.show(fig)
